In [1]:
# ============================================================
# CELL 1 — LOAD GITHUB REPOSITORY
# ============================================================

import os

REPO_DIR = "/content/TA"

if not os.path.exists(os.path.join(REPO_DIR, ".git")):
    !git clone https://github.com/shefiramarizcha62-sudo/TA.git /content/TA
else:
    !cd /content/TA && git pull origin main

print("=" * 70)
print("REPOSITORY READY")
print("=" * 70)

print(REPO_DIR)

Cloning into '/content/TA'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (47/47), done.
remote: Total 54 (delta 10), reused 40 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 1.24 MiB | 7.45 MiB/s, done.
Resolving deltas: 100% (10/10), done.
REPOSITORY READY
/content/TA


In [3]:
# ============================================================
# CELL 2 — LOAD REVISED DATASET
# ============================================================

import pandas as pd
import os

DATASET_PATH = os.path.join(
    REPO_DIR,
    "dataset",
    "dataset_preprocessed_C0-C4_revised.csv"
)

df = pd.read_csv(DATASET_PATH)

print("=" * 70)
print("REVISED DATASET")
print("=" * 70)

print("Path :", DATASET_PATH)
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

display(df.head())

REVISED DATASET
Path : /content/TA/dataset/dataset_preprocessed_C0-C4_revised.csv
Shape: (3815, 13)

Columns:
['No', 'Komentar', 'Label Awal', 'Label Pakar', 'Pasal Dominan', 'Catatan', 'Unnamed: 6', 'Kode Label', 'jumlah_kata', 'jumlah_karakter', 'clean_comment', 'Komentar_clean', 'Komentar_preprocessed']


,No,Komentar,Label Awal,Label Pakar,Pasal Dominan,Catatan,Unnamed: 6,Kode Label,jumlah_kata,jumlah_karakter,clean_comment,Komentar_clean,Komentar_preprocessed
0,5708.0,Apakah seorang maling anak'y jg akan menjadi m...,Indikasi penghinaan atau kebencian terhadap in...,Ambigu atau konteks tidak cukup,NaN,Pertanyaan umum tanpa target yang jelas.,NaN,C4,8.0,56.0,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anak'y jg akan menjadi m...,apakah seorang maling anaknya juga akan menjad...
1,495.0,"​@HanaFanda-wx1cciyaa betul,\nMeni hese ngesah...",Netral atau tidak terindikasi,Konten ofensif,NaN,Tidak terindikasi Pasal 27A maupun Pasal 28 ay...,NaN,C2,12.0,91.0,"​@hanafanda-wx1cciyaa betul, meni hese ngesahk...","​-wx1cciyaa betul, meni hese ngesahkeun pearam...","​-wx1cciyaa betul, meni hese ngesahkeun pearam..."
2,1841.0,Omongan nya pak bahlil sering kontradiksi. Lah...,Netral atau tidak terindikasi,Netral atau tidak terindikasi,NaN,NaN,NaN,C0,7.0,50.0,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...,omongan nya pak bahlil sering kontradiksi. lah...
3,6483.0,Astagfirullah. Katanya Wakil Rakyat tapi kelak...,Indikasi penghinaan atau kebencian terhadap in...,Konten ofensif,NaN,Menuduh wakil rakyat sebagai maling dan zalim.,NaN,C2,13.0,96.0,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tapi kelak...,astagfirullah. katanya wakil rakyat tetapi kel...
4,2222.0,"Cobalah pampang ke45 poin itu di media sosial,...",Netral atau tidak terindikasi,Ambigu atau konteks tidak cukup,NaN,komentar ini belum menunjukkan unsur yang cuku...,NaN,C4,12.0,76.0,"cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,...","cobalah pampang ke45 poin itu di media sosial,..."


In [6]:
# CELL 2 — SPLIT DATASET
from sklearn.model_selection import train_test_split

TEXT_COL = "Komentar_preprocessed"
LABEL_COL = "Kode Label"

# Hapus baris yang tidak memiliki teks atau label
df_model = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()

# Split 80% train, 10% validation, 10% test
train_df, temp_df = train_test_split(
    df_model,
    test_size=0.20,
    stratify=df_model[LABEL_COL],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df[LABEL_COL],
    random_state=42
)

print("="*70)
print("DATA SPLIT")
print("="*70)

print("Train :", train_df.shape)
print("Validation :", val_df.shape)
print("Test :", test_df.shape)

print("\nDistribusi Train:")
print(train_df[LABEL_COL].value_counts().sort_index())

print("\nDistribusi Validation:")
print(val_df[LABEL_COL].value_counts().sort_index())

print("\nDistribusi Test:")
print(test_df[LABEL_COL].value_counts().sort_index())

DATA SPLIT
Train : (3052, 13)
Validation : (381, 13)
Test : (382, 13)

Distribusi Train:
Kode Label
C0    610
C1    611
C2    610
C3    610
C4    611
Name: count, dtype: int64

Distribusi Validation:
Kode Label
C0    76
C1    76
C2    77
C3    76
C4    76
Name: count, dtype: int64

Distribusi Test:
Kode Label
C0    77
C1    76
C2    76
C3    77
C4    76
Name: count, dtype: int64


In [7]:
# CELL 3 — CHECK DATA LEAKAGE

train_texts = set(train_df[TEXT_COL].astype(str))
val_texts = set(val_df[TEXT_COL].astype(str))
test_texts = set(test_df[TEXT_COL].astype(str))

train_val_overlap = train_texts & val_texts
train_test_overlap = train_texts & test_texts
val_test_overlap = val_texts & test_texts

print("="*70)
print("DATA LEAKAGE CHECK")
print("="*70)

print("Train ∩ Validation :", len(train_val_overlap))
print("Train ∩ Test       :", len(train_test_overlap))
print("Validation ∩ Test  :", len(val_test_overlap))

if len(train_val_overlap) == 0 and len(train_test_overlap) == 0 and len(val_test_overlap) == 0:
    print("\n✓ Tidak ditemukan data yang overlap antar split.")
else:
    print("\n⚠ Ditemukan overlap. Perlu diperiksa sebelum training.")



DATA LEAKAGE CHECK
Train ∩ Validation : 0
Train ∩ Test       : 1
Validation ∩ Test  : 0

⚠ Ditemukan overlap. Perlu diperiksa sebelum training.


In [8]:
# CELL 4 — SAVE SPLITS

train_df.to_csv("slicing_train.csv", index=False)
val_df.to_csv("slicing_val.csv", index=False)
test_df.to_csv("slicing_test.csv", index=False)

print("="*70)
print("SPLIT DATASET TERSIMPAN")
print("="*70)

print("Train      :", len(train_df), "data")
print("Validation :", len(val_df), "data")
print("Test       :", len(test_df), "data")

print("\nFile:")
print("- slicing_train.csv")
print("- slicing_val.csv")
print("- slicing_test.csv")

SPLIT DATASET TERSIMPAN
Train      : 3052 data
Validation : 381 data
Test       : 382 data

File:
- slicing_train.csv
- slicing_val.csv
- slicing_test.csv


In [9]:
# CELL 5 — CONFIGURATION & TOKENIZER

import torch
from transformers import AutoTokenizer

MODEL_NAME = "indobenchmark/indobert-large-p2"

MAX_LENGTH = 256
SLICE_LENGTH = 128

LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
NUM_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 2

SEED = 42

print("="*70)
print("EXPERIMENT CONFIGURATION")
print("="*70)

print("Model               :", MODEL_NAME)
print("MAX_LENGTH          :", MAX_LENGTH)
print("Slice Length        :", SLICE_LENGTH)
print("Learning Rate       :", LEARNING_RATE)
print("Weight Decay        :", WEIGHT_DECAY)
print("Max Epochs          :", NUM_EPOCHS)
print("Early Stopping      :", EARLY_STOPPING_PATIENCE)
print("Seed                :", SEED)
print("Device              :", "cuda" if torch.cuda.is_available() else "cpu")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("\nTokenizer berhasil dimuat.")
print("Tokenizer:", tokenizer.__class__.__name__)

EXPERIMENT CONFIGURATION
Model               : indobenchmark/indobert-large-p2
MAX_LENGTH          : 256
Slice Length        : 128
Learning Rate       : 2e-05
Weight Decay        : 0.01
Max Epochs          : 10
Early Stopping      : 2
Seed                : 42
Device              : cuda


config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]


Tokenizer berhasil dimuat.
Tokenizer: BertTokenizer


In [10]:
# CELL 6 — TOKEN SLICING

def slice_text(text, tokenizer, slice_length=128):
    """
    Membagi teks menjadi beberapa potongan berdasarkan token.
    Special tokens tidak disertakan pada tahap slicing.
    """
    text = str(text)

    # Tokenisasi tanpa special tokens
    tokens = tokenizer.encode(
        text,
        add_special_tokens=False,
        truncation=False
    )

    # Jika teks kosong
    if len(tokens) == 0:
        return [""]

    slices = []

    # Membagi token menjadi potongan 128
    for i in range(0, len(tokens), slice_length):
        chunk_tokens = tokens[i:i + slice_length]

        # Kembalikan token menjadi teks
        chunk_text = tokenizer.decode(
            chunk_tokens,
            skip_special_tokens=True
        )

        slices.append(chunk_text)

    return slices


# Contoh sebelum diterapkan ke seluruh dataset
sample_text = train_df[TEXT_COL].iloc[0]

sample_slices = slice_text(
    sample_text,
    tokenizer,
    SLICE_LENGTH
)

print("="*70)
print("CONTOH SLICING")
print("="*70)

print("Teks asli:")
print(sample_text)

print("\nJumlah slice:", len(sample_slices))

for i, s in enumerate(sample_slices):
    print(f"\nSlice {i+1}:")
    print(s)

CONTOH SLICING
Teks asli:
wah sudah parah kesal ini roman2 nya bang fery sampai gemeteran cuy bicara nya

Jumlah slice: 1

Slice 1:
wah sudah parah kesal ini roman2 nya bang fery sampai gemeteran cuy bicara nya


In [11]:
# CELL 7 — APPLY SLICING TO TRAIN, VALIDATION, TEST

def create_sliced_dataset(dataframe):
    rows = []

    for original_index, row in dataframe.reset_index(drop=True).iterrows():
        text = row[TEXT_COL]
        label = row[LABEL_COL]

        slices = slice_text(
            text,
            tokenizer,
            SLICE_LENGTH
        )

        for slice_id, sliced_text in enumerate(slices):
            rows.append({
                "original_index": original_index,
                "slice_id": slice_id,
                "text": sliced_text,
                "label": label
            })

    return pd.DataFrame(rows)


# Terapkan slicing
train_sliced = create_sliced_dataset(train_df)
val_sliced = create_sliced_dataset(val_df)
test_sliced = create_sliced_dataset(test_df)


print("="*70)
print("HASIL SLICING")
print("="*70)

print("Train original :", len(train_df))
print("Train sliced   :", len(train_sliced))

print("\nValidation original :", len(val_df))
print("Validation sliced   :", len(val_sliced))

print("\nTest original :", len(test_df))
print("Test sliced   :", len(test_sliced))

print("\nContoh hasil slicing:")
display(train_sliced.head(10))

HASIL SLICING
Train original : 3052
Train sliced   : 3168

Validation original : 381
Validation sliced   : 406

Test original : 382
Test sliced   : 405

Contoh hasil slicing:


,original_index,slice_id,text,label
0,0,0,wah sudah parah kesal ini roman2 nya bang fery...,C0
1,1,0,rakyat khawatir ternyata prabowo sejenis dia.,C4
2,2,0,ah pak pigai sudah banyak minum susu ini dan m...,C3
3,3,0,terus apa hubungan kewaspadaan dengan keturuna...,C0
4,4,0,yang bikin uu membolehkan siapa mirip pendukun...,C1
5,5,0,apa guna uud kalau hukuman mati sama perampasa...,C2
6,6,0,gog gog gog odgj lagi gong gong,C4
7,7,0,"mantap grace natalie, sukses terus untuk psi.",C0
8,8,0,apa mendikbud tidak ikut bersalah dengan membi...,C1
9,9,0,kalau sampai 3 priode rakyat tambah pecah.,C1


In [12]:
# CELL 8 — LABEL DISTRIBUTION AFTER SLICING

print("="*70)
print("LABEL DISTRIBUTION AFTER SLICING")
print("="*70)

print("\nTrain:")
print(train_sliced["label"].value_counts().sort_index())

print("\nValidation:")
print(val_sliced["label"].value_counts().sort_index())

print("\nTest:")
print(test_sliced["label"].value_counts().sort_index())

print("\nPersentase Train:")
print(
    (train_sliced["label"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)

LABEL DISTRIBUTION AFTER SLICING

Train:
label
C0    615
C1    633
C2    632
C3    677
C4    611
Name: count, dtype: int64

Validation:
label
C0    80
C1    78
C2    80
C3    91
C4    77
Name: count, dtype: int64

Test:
label
C0    79
C1    85
C2    77
C3    87
C4    77
Name: count, dtype: int64

Persentase Train:
label
C0    19.41
C1    19.98
C2    19.95
C3    21.37
C4    19.29
Name: proportion, dtype: float64


In [13]:
# CELL 9 — TOKENIZATION

def tokenize_dataset(dataframe):
    return tokenizer(
        dataframe["text"].tolist(),
        padding=False,
        truncation=True,
        max_length=MAX_LENGTH
    )


train_encodings = tokenize_dataset(train_sliced)
val_encodings = tokenize_dataset(val_sliced)
test_encodings = tokenize_dataset(test_sliced)

print("="*70)
print("TOKENIZATION")
print("="*70)

print("Train samples :", len(train_encodings["input_ids"]))
print("Validation samples :", len(val_encodings["input_ids"]))
print("Test samples :", len(test_encodings["input_ids"]))

print("\nContoh:")
print("Jumlah token:", len(train_encodings["input_ids"][0]))
print("Input IDs:", train_encodings["input_ids"][0][:20])
print("Tokens:", tokenizer.convert_ids_to_tokens(
    train_encodings["input_ids"][0][:20]
))

TOKENIZATION
Train samples : 3168
Validation samples : 406
Test samples : 405

Contoh:
Jumlah token: 21
Input IDs: [2, 2282, 259, 4762, 10531, 92, 18449, 30378, 1107, 1719, 6028, 30371, 493, 5536, 2194, 5, 4250, 30371, 4007, 1107]
Tokens: ['[CLS]', 'wah', 'sudah', 'parah', 'kesal', 'ini', 'roman', '##2', 'nya', 'bang', 'fer', '##y', 'sampai', 'gem', '##eter', '##an', 'cu', '##y', 'bicara', 'nya']


In [14]:
# CELL 10 — CREATE HUGGING FACE DATASET

from datasets import Dataset

# Tambahkan label numerik C0-C4
label2id = {
    "C0": 0,
    "C1": 1,
    "C2": 2,
    "C3": 3,
    "C4": 4
}

id2label = {v: k for k, v in label2id.items()}

def build_hf_dataset(dataframe, encodings):
    dataset_dict = {
        "input_ids": encodings["input_ids"],
        "attention_mask": encodings["attention_mask"],
        "original_index": dataframe["original_index"].tolist(),
        "slice_id": dataframe["slice_id"].tolist(),
        "labels": [
            label2id[label]
            for label in dataframe["label"].tolist()
        ]
    }

    return Dataset.from_dict(dataset_dict)


train_dataset = build_hf_dataset(train_sliced, train_encodings)
val_dataset = build_hf_dataset(val_sliced, val_encodings)
test_dataset = build_hf_dataset(test_sliced, test_encodings)

print("="*70)
print("HUGGING FACE DATASET")
print("="*70)

print("Train      :", train_dataset)
print("Validation :", val_dataset)
print("Test       :", test_dataset)

print("\nContoh data:")
print(train_dataset[0])

HUGGING FACE DATASET
Train      : Dataset({
    features: ['input_ids', 'attention_mask', 'original_index', 'slice_id', 'labels'],
    num_rows: 3168
})
Validation : Dataset({
    features: ['input_ids', 'attention_mask', 'original_index', 'slice_id', 'labels'],
    num_rows: 406
})
Test       : Dataset({
    features: ['input_ids', 'attention_mask', 'original_index', 'slice_id', 'labels'],
    num_rows: 405
})

Contoh data:
{'input_ids': [2, 2282, 259, 4762, 10531, 92, 18449, 30378, 1107, 1719, 6028, 30371, 493, 5536, 2194, 5, 4250, 30371, 4007, 1107, 3], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'original_index': 0, 'slice_id': 0, 'labels': 0}


In [15]:
# CELL 11 — DATA COLLATOR

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True
)

print("="*70)
print("DATA COLLATOR")
print("="*70)

print("Dynamic padding: aktif")
print("MAX_LENGTH:", MAX_LENGTH)

DATA COLLATOR
Dynamic padding: aktif
MAX_LENGTH: 256


In [22]:
# CELL 12 — LOAD INDOBERT LARGE P2

from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

print("="*70)
print("MODEL")
print("="*70)

print("Model :", MODEL_NAME)
print("Number of labels :", model.config.num_labels)
print("Labels :", model.config.id2label)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.34GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.34GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-large-p2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MODEL
Model : indobenchmark/indobert-large-p2
Number of labels : 5
Labels : {0: 'C0', 1: 'C1', 2: 'C2', 3: 'C3', 4: 'C4'}


In [16]:
# CELL 13 — EVALUATION METRICS

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

print("="*70)
print("METRICS")
print("="*70)

print("✓ Accuracy")
print("✓ Macro Precision")
print("✓ Macro Recall")
print("✓ Macro F1")

METRICS
✓ Accuracy
✓ Macro Precision
✓ Macro Recall
✓ Macro F1


In [17]:
# CELL 14 — CLASS WEIGHT

import torch
import torch.nn as nn
from transformers import Trainer

# Hitung jumlah data per kelas pada TRAIN setelah slicing
label_counts = train_sliced["label"].map(label2id).value_counts().sort_index()

class_weights = len(train_sliced) / (
    len(label_counts) * label_counts
)

class_weights = torch.tensor(
    class_weights.values,
    dtype=torch.float
)

print("="*70)
print("CLASS WEIGHTS")
print("="*70)

for class_id, weight in enumerate(class_weights):
    print(f"{id2label[class_id]} : {weight:.4f}")

CLASS WEIGHTS
C0 : 1.0302
C1 : 1.0009
C2 : 1.0025
C3 : 0.9359
C4 : 1.0370


In [18]:
# CELL 15 — WEIGHTED TRAINER

class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss


print("="*70)
print("WEIGHTED TRAINER")
print("="*70)
print("✓ CrossEntropyLoss dengan class weight aktif")

WEIGHTED TRAINER
✓ CrossEntropyLoss dengan class weight aktif


In [19]:
# CELL 16 — TRAINING ARGUMENTS

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results_slicing",

    # Training
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=NUM_EPOCHS,

    # Batch
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    # Evaluation
    eval_strategy="epoch",
    save_strategy="epoch",

    # Select best model
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # Early stopping
    logging_strategy="epoch",

    # Reproducibility
    seed=SEED,

    # Performance
    fp16=torch.cuda.is_available(),

    # Disable external logging
    report_to="none",

    # Save
    save_total_limit=2
)

print("="*70)
print("TRAINING ARGUMENTS")
print("="*70)

print("Learning rate       :", LEARNING_RATE)
print("Weight decay        :", WEIGHT_DECAY)
print("Train batch size    : 4")
print("Eval batch size     : 4")
print("Max epochs          :", NUM_EPOCHS)
print("Early stopping      :", EARLY_STOPPING_PATIENCE)
print("Evaluation          : epoch")
print("Best metric         : Macro-F1")
print("FP16                :", torch.cuda.is_available())
print("Seed                :", SEED)

TRAINING ARGUMENTS
Learning rate       : 2e-05
Weight decay        : 0.01
Train batch size    : 4
Eval batch size     : 4
Max epochs          : 10
Early stopping      : 2
Evaluation          : epoch
Best metric         : Macro-F1
FP16                : True
Seed                : 42


In [24]:
# CELL 17 — CREATE TRAINER WITH EARLY STOPPING

from transformers import EarlyStoppingCallback

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE
        )
    ]
)

print("="*70)
print("TRAINER READY")
print("="*70)

print("Model              :", MODEL_NAME)
print("Train samples      :", len(train_dataset))
print("Validation samples :", len(val_dataset))
print("Batch size         : 4")
print("Early stopping     :", EARLY_STOPPING_PATIENCE)
print("Best metric        : Macro-F1")
print("Class weight       : Active")

TRAINER READY
Model              : indobenchmark/indobert-large-p2
Train samples      : 3168
Validation samples : 406
Batch size         : 4
Early stopping     : 2
Best metric        : Macro-F1
Class weight       : Active


In [25]:
# CELL 18 — TRAINING

print("="*70)
print("START TRAINING — SLICING EXPERIMENT")
print("="*70)

train_result = trainer.train()

print("\n" + "="*70)
print("TRAINING SELESAI")
print("="*70)

print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation Macro-F1:", trainer.state.best_metric)

START TRAINING — SLICING EXPERIMENT


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Macro F1
1,1.425558,1.271641,0.440887,0.486984,0.444427,0.431661
2,0.970733,1.280279,0.517241,0.509982,0.514480,0.501807
3,0.458239,1.873759,0.504926,0.520697,0.502929,0.504359
4,0.192029,2.967630,0.517241,0.559902,0.517523,0.523998
5,0.083375,3.626179,0.470443,0.492759,0.474555,0.469557
6,0.061466,3.547693,0.487685,0.508044,0.488031,0.482190


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


TRAINING SELESAI
Best checkpoint: ./results_slicing/checkpoint-3168
Best validation Macro-F1: 0.5239978559442331
